[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_15/SFM_ch15_covar/SFM_ch15_covar.ipynb)

# SFM_ch15_covar

CoVaR and Delta-CoVaR (Adrian and Brunnermeier, 2016) at 1% and 5% by linear quantile regression (Koenker and Bassett, 1978) of the regional equity index (S&P 500, Euro Stoxx 50, BET) on the return of each of 13 banks, with 95% moving-block bootstrap intervals (blocks of 20 days, 200 resamples); the quantile regression lines of JPMorgan Chase and the S&P 500; VaR 1% of the bank against Delta-CoVaR 1%; a time-varying Delta-CoVaR with the lagged VIX level and index return as state variables.

*Statistics of Financial Markets (SFM), Chapter 15: Systemic risk. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_15'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
import matplotlib.dates as mdates

import statsmodels.api as sm

from statsmodels.regression.quantile_regression import QuantReg

from scipy.special import gammaln

START, END = '2005-01-01', '2026-09-18'

BANKS = {'JPM': ('JPM.US', 'JPMorgan Chase', 'US'), 'BAC': ('BAC.US', 'Bank of America', 'US'), 'C': ('C.US', 'Citigroup', 'US'), 'GS': ('GS.US', 'Goldman Sachs', 'US'), 'MS': ('MS.US', 'Morgan Stanley', 'US'), 'WFC': ('WFC.US', 'Wells Fargo', 'US'), 'DBK': ('DBK.XETRA', 'Deutsche Bank', 'EU'), 'BNP': ('BNP.PA', 'BNP Paribas', 'EU'), 'SAN': ('SAN.MC', 'Santander', 'EU'), 'INGA': ('INGA.AS', 'ING', 'EU'), 'HSBA': ('HSBA.LSE', 'HSBC', 'EU'), 'TLV': ('TLV.RO', 'Banca Transilvania', 'RO'), 'BRD': ('BRD.RO', 'BRD', 'RO')}

REGIONS = {'US': ['JPM', 'BAC', 'C', 'GS', 'MS', 'WFC'], 'EU': ['DBK', 'BNP', 'SAN', 'INGA', 'HSBA'], 'RO': ['TLV', 'BRD']}

ALL = ['JPM', 'BAC', 'C', 'GS', 'MS', 'WFC', 'DBK', 'BNP', 'SAN', 'INGA', 'HSBA', 'TLV', 'BRD']

INTL = ['JPM', 'BAC', 'C', 'GS', 'MS', 'WFC', 'DBK', 'BNP', 'SAN', 'INGA', 'HSBA']

REGION_START = {'US': '2005-01-01', 'EU': '2005-01-01', 'RO': '2010-01-01'}

INDEX = {'US': 'sp500', 'EU': 'stoxx50', 'RO': 'bet'}

INDEX_NAME = {'sp500': 'S&P 500', 'stoxx50': 'Euro Stoxx 50', 'bet': 'BET'}

REG_COL = {'US': '#1A3A6E', 'EU': '#CD0000', 'RO': '#2E7D32'}

REG_LAB = {'US': 'US banks', 'EU': 'European banks', 'RO': 'Romanian banks (TLV, BRD, since 2010)'}

BAD_DAYS = {'TLV': ['2016-05-30', '2016-05-31']}

EVENTS = [('2008-09-15', 'Lehman'), ('2012-07-26', 'Draghi'), ('2020-03-16', 'COVID-19'), ('2023-03-10', 'SVB')]

EPISODES = {'Lehman 2008': ('2008-08-29', '2008-12-31'), 'Euro area 2011-2012': ('2011-06-30', '2012-09-28'), 'March 2020': ('2020-02-14', '2020-04-30'), 'March 2023': ('2023-02-28', '2023-04-28')}

ALPHA = 0.01

ALPHA_MES = 0.05

K_CAP = 0.08

B_BOOT, BLOCK = 200, 20

GC_WINDOW = 36

DY_H, DY_WINDOW, DY_STEP = 10, 200, 5

SEED = 2026

def bank_price(k, start=START, end=END):
    """Daily adjusted close of a bank (weekdays; no unchanged holiday-filled closes; BVB: no days without trades)."""
    sym, _, reg = BANKS[k]
    d = read_market(sym).loc[start:end]
    d = d[(d.index.dayofweek < 5) & (d['adjusted_close'] > 0)]
    if reg == 'RO' and 'volume' in d.columns:
        d = d[d['volume'] > 0]
    s = d['adjusted_close'].astype(float)
    return s[s.diff() != 0].rename(k)


def prices(keys, start=START, end=END):
    """Banks and indices (sp500, stoxx50, bet, vix) on their common trading days."""
    cols = [bank_price(k, start, end) if k in BANKS else load_close(k, start, end).rename(k) for k in keys]
    return pd.concat(cols, axis=1, join='inner').dropna()


def returns(keys, start=START, end=END):
    """Daily log returns in %: prices joined on common days first, then returns; known data errors removed."""
    R = 100 * np.log(prices(keys, start, end)).diff().dropna()
    for k, days in BAD_DAYS.items():
        if k in R.columns:
            R = R.drop(pd.to_datetime(days), errors='ignore')
    return R


def region_returns(reg, start=None, end=END):
    """The banks of a region and its equity index (S&P 500, Euro Stoxx 50 or BET), daily log returns in %."""
    return returns(REGIONS[reg] + [INDEX[reg]], max(start or START, REGION_START[reg]), end)


def bank_portfolio(R, members):
    """Equal-weighted bank portfolio (daily rebalancing): log return in % of the average simple return."""
    return 100 * np.log(1 + (np.exp(R[members] / 100) - 1).mean(axis=1))


def qreg(y, X, q):
    """Linear quantile regression (Koenker and Bassett, 1978): coefficients [intercept, slopes]."""
    return QuantReg(np.asarray(y), sm.add_constant(np.asarray(X), has_constant='add')).fit(q=q, max_iter=5000).params


def covar(x_sys, x_i, alpha=ALPHA):
    """CoVaR and Delta-CoVaR (%, losses positive): quantile regression x_sys = a + b x_i at level alpha;
    CoVaR = -(a + b q_alpha(x_i)); Delta-CoVaR = b (q_50%(x_i) - q_alpha(x_i))."""
    x_sys, x_i = np.asarray(x_sys, float), np.asarray(x_i, float)
    a, b = qreg(x_sys, x_i, alpha)
    qa, qm = np.quantile(x_i, alpha), np.quantile(x_i, 0.5)
    return {'a': float(a), 'b': float(b), 'q_a': float(qa), 'q_m': float(qm), 'var_i': float(-qa),
            'covar': float(-(a + b * qa)), 'covar_med': float(-(a + b * qm)), 'dcovar': float(b * (qm - qa)),
            'var_sys': float(-np.quantile(x_sys, alpha))}


def block_idx(n, block, rng):
    """Indices of a moving-block bootstrap sample (blocks of `block` consecutive days)."""
    s = rng.integers(0, n - block + 1, size=int(np.ceil(n / block)))
    return (s[:, None] + np.arange(block)[None, :]).ravel()[:n]


def covar_ci(x_sys, x_i, alpha=ALPHA, B=B_BOOT, block=BLOCK, seed=SEED):
    """95% percentile interval of Delta-CoVaR by moving-block bootstrap."""
    x_sys, x_i = np.asarray(x_sys, float), np.asarray(x_i, float)
    rng = np.random.default_rng(seed)
    d = [covar(x_sys[i], x_i[i], alpha)['dcovar'] for i in (block_idx(len(x_i), block, rng) for _ in range(B))]
    return [float(v) for v in np.percentile(d, [2.5, 97.5])]


def covar_table(boot=True):
    """CoVaR 1% and 5% of the regional index conditional on each bank (2005-2026)."""
    rows = {}
    for reg in ('US', 'EU', 'RO'):
        R = region_returns(reg)
        for k in REGIONS[reg]:
            c1 = covar(R[INDEX[reg]], R[k], 0.01)
            c5 = covar(R[INDEX[reg]], R[k], 0.05)
            rows[k] = {'region': reg, 'n': len(R), **c1, 'dcovar5': c5['dcovar'], 'b5': c5['b'],
                       'ci': covar_ci(R[INDEX[reg]], R[k]) if boot else [np.nan, np.nan]}
    return rows


def fig_covar_qr(save=True):
    """JPMorgan Chase and the S&P 500: quantile regression lines at 1% and 50%, the 1% and median days of JPM."""
    R = region_returns('US')
    x, y = R['JPM'], R['sp500']
    c = covar(y, x, 0.01)
    a50, b50 = qreg(y, x, 0.5)
    fig, ax = plt.subplots(figsize=(10, 4.4))
    ax.scatter(x, y, s=5, color=st.MainBlue, alpha=0.35, label='daily returns, JPMorgan Chase and S&P 500')
    g = np.linspace(x.min(), x.max(), 100)
    ax.plot(g, c['a'] + c['b'] * g, color=st.IDAred, lw=2, label=f'1% quantile regression: {c["a"]:.2f} + {c["b"]:.2f} x')
    ax.plot(g, a50 + b50 * g, color=st.Forest, lw=2, label=f'median regression: {a50:.2f} + {b50:.2f} x')
    ax.axvline(c['q_a'], color=st.Orange, ls='--', lw=1.3, label=f'1% quantile of JPM: {c["q_a"]:.2f}%')
    ax.axvline(c['q_m'], color=st.Purple, ls='--', lw=1.3, label=f'median of JPM: {c["q_m"]:.2f}%')
    ax.plot([c['q_a']], [-c['covar']], 'o', color=st.IDAred, ms=8, label=f'minus CoVaR 1%: {-c["covar"]:.2f}%')
    ax.plot([c['q_m']], [-c['covar_med']], 's', color=st.Forest, ms=8, label=f'minus CoVaR 1% at the median: {-c["covar_med"]:.2f}%')
    ax.set_xlim(-12, 12)
    ax.set_ylim(-13, 8)
    ax.set_xlabel('JPMorgan Chase daily log return (%)')
    ax.set_ylabel('S&P 500 daily log return (%)')
    st.legend_outside_bottom(ax, ncol=2, y=-0.16)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch15_covar_qr')
    return {**c, 'a50': float(a50), 'b50': float(b50), 'n': len(R), 'start': R.index[0].date().isoformat()}


def fig_dcovar(t, save=True):
    """Delta-CoVaR 1% of each bank with its 95% block-bootstrap interval."""
    fig, ax = plt.subplots(figsize=(11, 3.9))
    ks = list(t)
    xs = np.arange(len(ks))
    v = np.array([t[k]['dcovar'] for k in ks])
    lo = np.array([t[k]['ci'][0] for k in ks])
    hi = np.array([t[k]['ci'][1] for k in ks])
    ax.bar(xs, v, color=[REG_COL[t[k]['region']] for k in ks], width=0.62)
    ax.errorbar(xs, v, yerr=[v - lo, hi - v], fmt='none', ecolor='black', capsize=3, lw=1)
    ax.set_xticks(xs)
    ax.set_xticklabels([BANKS[k][1] for k in ks], rotation=30, ha='right', fontsize=10.5)
    ax.set_ylabel('Delta-CoVaR 1% (%)')
    h = [plt.Rectangle((0, 0), 1, 1, color=REG_COL[r]) for r in ('US', 'EU', 'RO')]
    lab = ['US banks (system: S&P 500)', 'European banks (system: Euro Stoxx 50)', 'Romanian banks (system: BET)']
    h.append(plt.Line2D([], [], color='black', marker='|', ls='', ms=10))
    lab.append('95% block-bootstrap interval')
    ax.legend(h, lab, loc='upper center', bbox_to_anchor=(0.5, -0.36), ncol=2, frameon=False)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch15_dcovar')


def fig_var_vs_dcovar(t, save=True):
    """VaR 1% of each bank against its Delta-CoVaR 1%."""
    fig, ax = plt.subplots(figsize=(10, 4.2))
    for reg in ('US', 'EU', 'RO'):
        ks = [k for k in t if t[k]['region'] == reg]
        ax.scatter([t[k]['var_i'] for k in ks], [t[k]['dcovar'] for k in ks], s=50, color=REG_COL[reg], label=REG_LAB[reg])
        for k in ks:
            ax.annotate(k, (t[k]['var_i'], t[k]['dcovar']), xytext=(4, 3), textcoords='offset points', fontsize=10, color='black')
    ax.set_xlabel('VaR 1% of the bank (%)')
    ax.set_ylabel('Delta-CoVaR 1% (%)')
    st.legend_outside_bottom(ax, ncol=3, y=-0.17)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch15_var_vs_dcovar')
    rho = stats.spearmanr([t[k]['var_i'] for k in t], [t[k]['dcovar'] for k in t])[0]
    return float(rho)


def dynamic_dcovar(k, alpha=ALPHA):
    """Time-varying Delta-CoVaR (Adrian and Brunnermeier, 2016, Section II) with the lagged state variables
    M_{t-1} = (VIX level, index return): quantile regressions x_i = c + g'M (levels alpha and 50%) and
    x_sys = a + b x_i + g'M (level alpha); Delta-CoVaR_t = b (q50_t - qalpha_t)."""
    reg = BANKS[k][2]
    R = region_returns(reg)
    vix = load_close('vix', START, END)
    D = pd.concat([R[[k, INDEX[reg]]], vix.reindex(R.index).ffill().rename('vix')], axis=1).dropna()
    M = pd.concat([D['vix'].shift(1), D[INDEX[reg]].shift(1)], axis=1).iloc[1:]
    D = D.iloc[1:]
    ca, cm = qreg(D[k], M, alpha), qreg(D[k], M, 0.5)
    cs = qreg(D[INDEX[reg]], np.column_stack([D[k], M]), alpha)
    Mc = sm.add_constant(M.values, has_constant='add')
    qa, qm = Mc @ ca, Mc @ cm
    return pd.Series(cs[1] * (qm - qa), index=D.index, name=k), float(cs[1])


def fig_dcovar_dynamic(save=True):
    """Time-varying Delta-CoVaR 1% of JPMorgan Chase, Deutsche Bank and Banca Transilvania."""
    fig, ax = plt.subplots(figsize=(11, 3.8))
    out = {}
    for k, c in [('JPM', REG_COL['US']), ('DBK', REG_COL['EU']), ('TLV', REG_COL['RO'])]:
        s, b = dynamic_dcovar(k)
        ax.plot(s.index, s, color=c, lw=0.9, label=f'{BANKS[k][1]}')
        out[k] = {'mean': float(s.mean()), 'max': float(s.max()), 'date_max': s.idxmax().date().isoformat(),
                  'b': b, 'y2017': float(s.loc['2017'].mean()), 'last': float(s.iloc[-1])}
    for d, lab in EVENTS:
        ax.axvline(pd.Timestamp(d), color=st.Purple, ls=':', lw=1.0, label='_')
    ax.set_ylabel('Delta-CoVaR 1% (%)')
    st.legend_outside_bottom(ax, ncol=3, y=-0.13)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch15_dcovar_dynamic')
    return out

## Run

In [ ]:
print(fig_covar_qr())
CT = covar_table()
print(pd.DataFrame(CT).T[['region', 'b', 'var_i', 'covar', 'dcovar', 'dcovar5', 'ci']])
fig_dcovar(CT)
print(fig_var_vs_dcovar(CT))
print(fig_dcovar_dynamic())

![sfm_ch15_covar_qr](./sfm_ch15_covar_qr.png)

Output: `sfm_ch15_covar_qr.pdf`

![sfm_ch15_dcovar](./sfm_ch15_dcovar.png)

Output: `sfm_ch15_dcovar.pdf`

![sfm_ch15_var_vs_dcovar](./sfm_ch15_var_vs_dcovar.png)

Output: `sfm_ch15_var_vs_dcovar.pdf`

![sfm_ch15_dcovar_dynamic](./sfm_ch15_dcovar_dynamic.png)

Output: `sfm_ch15_dcovar_dynamic.pdf`